In [1]:
# --- BLOQUE 0: ACTUALIZACIÓN DE INFRAESTRUCTURA ---
import sys
import subprocess
import warnings

warnings.filterwarnings('ignore')

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package])

try:
    import astroquery
    print(f"[SYSTEM] Astroquery version: {astroquery.__version__}")
    from astroquery.heasarc import Heasarc
    import astropy
    from astropy.coordinates import SkyCoord
    import scipy
    import matplotlib
except ImportError:
    print("[SYSTEM] Actualizando arsenal astronómico...")
    install("astroquery")
    install("astropy")
    install("scipy")
    install("matplotlib")
    from astroquery.heasarc import Heasarc
    from astropy.coordinates import SkyCoord

import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo
from scipy.optimize import curve_fit
import astropy.units as u

print("[SYSTEM] Iniciando Protocolo ASTRO-MINER v7 (Whole Sky Dragnet)...")

# --- BLOQUE 1: EXTRACCIÓN SOBERANA (RED DE ARRASTRE) ---
heasarc = Heasarc()
TARGET_TABLE = 'fermilgrb'

print(f"[QUERY] Ejecutando búsqueda de cono completo (180 deg) en {TARGET_TABLE}...")

try:
    # TÁCTICA: En lugar de buscar por nombre (que falla), buscamos por posición.
    # Centro: (0,0) Galáctico. Radio: 180 grados (Todo el cielo).
    # Esto obliga a HEASARC a devolver todo el catálogo.
    pos = SkyCoord(0, 0, unit="deg", frame='galactic')
    table = heasarc.query_region(pos, mission=TARGET_TABLE, radius='180 deg', result_max=3000)

    df = table.to_pandas()
    # Normalizar columnas
    df.columns = [c.upper() for c in df.columns]
    print(f"[SUCCESS] Tabla recuperada. Registros: {len(df)}")

except Exception as e:
    print(f"[ERROR] Fallo en búsqueda espacial: {e}")
    print("[RETRY] Intentando búsqueda por objeto 'GRB%'...")
    try:
        # Fallback a búsqueda por nombre explícito
        table = heasarc.query_object(object_name='GRB%', mission=TARGET_TABLE, result_max=3000)
        df = table.to_pandas()
        df.columns = [c.upper() for c in df.columns]
        print(f"[SUCCESS] Tabla recuperada por nombre. Registros: {len(df)}")
    except Exception as e2:
        print(f"[CRITICAL] HEASARC inaccesible. Error final: {e2}")
        sys.exit()

# --- BLOQUE 2: MAPEO DE COLUMNAS (Validado en v6) ---
print("[MAP] Mapeando variables físicas...")

try:
    # Usamos las columnas confirmadas en tus logs anteriores
    raw_z = df['REDSHIFT']
    # Energía máxima de emisión extendida (donde viven los fotones duros)
    raw_e = df['EXT_EMISSION_MAX_ENE']     # MeV
    # Tiempo de llegada de ese fotón
    raw_t = df['EXT_EMISSION_MAX_ENE_T']   # s
    raw_n = df['NAME']

except KeyError as e:
    print(f"[ERROR] Columna faltante: {e}")
    print(f"[DEBUG] Columnas disponibles: {list(df.columns)}")
    sys.exit()

# --- BLOQUE 3: FÍSICA DE ALTA ENERGÍA ---
# Convertimos MeV a GeV
raw_e_gev = raw_e / 1000.0

# Filtros de Calidad (Stricto Sensu)
# 1. Redshift > 0 (Distancia conocida)
# 2. Energía > 1 GeV (Para ver efectos cuánticos)
# 3. Tiempo definido
mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 1.0) & (np.isfinite(raw_t))

clean_z = raw_z[mask].values
clean_t = raw_t[mask].values
clean_e = raw_e_gev[mask].values
clean_names = raw_n[mask].values

print(f"[DATA] Eventos válidos para LIV (z>0, E>1GeV): {len(clean_z)}")

if len(clean_z) < 3:
    print("[WARN] Pocos datos. Esto es normal si el catálogo es estricto con 'Emisión Extendida'.")
    # Continuamos para ver qué sale, aunque sean pocos puntos.

# Cálculo K-Factor (Integral Cosmológica)
print("[PHYS] Integrando expansión del universo...")
def get_k(z):
    z_grid = np.linspace(0, z, 50)
    # H(z)/H0
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    # K_jacob_piran = integral (1+z)/H(z)
    integrand = (1+z_grid) / Ez
    return np.trapz(integrand, z_grid)

if len(clean_z) > 0:
    K_factors = np.array([get_k(z) for z in clean_z])
    # Observable Clave: Retraso por unidad de energía (s/GeV)
    # Si hay LIV, esto debe correlacionarse con K
    lags = clean_t / clean_e
else:
    K_factors = []
    lags = []

# --- BLOQUE 4: ANÁLISIS Y REGRESIÓN ---
def model(k, m, c):
    return m * k + c

if len(K_factors) > 2:
    # Usamos sigma para dar menos peso a los outliers ruidosos
    popt, pcov = curve_fit(model, K_factors, lags)
    m_fit = popt[0]
    m_err = np.sqrt(np.diag(pcov))[0]
else:
    m_fit, m_err = 0, 0
    print("[SKIP] Insuficientes datos para ajuste.")

# --- BLOQUE 5: VISUALIZACIÓN CIENTÍFICA ---
plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

if len(K_factors) > 0:
    # Scatter plot con color por energía
    sc = plt.scatter(K_factors, lags, c=clean_e, cmap='plasma', s=100, alpha=0.9, label='GRB Fotones')
    plt.colorbar(sc, label='Energía (GeV)')

    # Barras de error estimadas (10% sistemático)
    plt.errorbar(K_factors, lags, yerr=0.1*np.abs(lags)+0.5, fmt='none', ecolor='gray', alpha=0.5)

    if len(K_factors) > 2:
        x_plot = np.linspace(0, max(K_factors)*1.1, 50)
        plt.plot(x_plot, model(x_plot, m_fit, popt[1]), color='#00ffcc',
                 linestyle='--', lw=2, label=f'Ajuste LIV (m={m_fit:.2e})')

    plt.axhline(0, color='white', linestyle=':', label='Einstein (m=0)')

    # Etiquetas inteligentes
    for i, name in enumerate(clean_names):
        # Etiquetar solo si tiene alto retraso o alta energía
        if clean_e[i] > 5.0 or abs(lags[i]) > 2:
            plt.annotate(f"{name}\n{clean_e[i]:.1f}GeV",
                         (K_factors[i], lags[i]), color='yellow', fontsize=9,
                         xytext=(5, 5), textcoords='offset points')

plt.xlabel(r'Factor de Distancia Cosmológica $K(z)$')
plt.ylabel(r'Retraso Normalizado $\tau_n = \Delta t / E$ (s/GeV)')
plt.title('Cazando Violación de Lorentz (v7: Whole Sky)\nDatos: HEASARC fermilgrb [EXT_EMISSION]', color='white')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n[RESULT] Pendiente LIV: m = {m_fit:.4e} +/- {m_err:.4e} s/GeV")
# Interpretación
H0_inv_s = (1/cosmo.H0).to(u.s).value
if m_fit > 0:
    M_QG_est = H0_inv_s / m_fit # GeV aproximadamente
    print(f"[FISICA] Escala QG sugerida: ~{M_QG_est:.2e} GeV")

[SYSTEM] Actualizando arsenal astronómico...
[SYSTEM] Iniciando Protocolo ASTRO-MINER v7 (Whole Sky Dragnet)...
[QUERY] Ejecutando búsqueda de cono completo (180 deg) en fermilgrb...
[ERROR] Fallo en búsqueda espacial: Error[s] detected in query preprocessing. ERROR: radius must not be greater than 90 degrees or less than 0. Where: function &quot;circle&quot;
[RETRY] Intentando búsqueda por objeto 'GRB%'...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



[CRITICAL] HEASARC inaccesible. Error final: Unable to find coordinates for name 'GRB%' using https://cds.unistra.fr/cgi-bin/nph-sesame/SNV?GRB%25
Traceback (most recent call last):
  File "/tmp/ipython-input-2285528767.py", line 47, in <cell line: 0>
    table = heasarc.query_region(pos, mission=TARGET_TABLE, radius='180 deg', result_max=3000)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astropy/utils/decorators.py", line 620, in wrapper
    return function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astroquery/heasarc/core.py", line 424, in query_region
    response = self.query_tap(query=adql, maxrec=maxrec)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astroquery/heasarc/core.py", line 289, in query_tap
    return self.tap.search(query, language='ADQL', maxrec=maxrec)
  

TypeError: object of type 'NoneType' has no len()